# GEF2017 当前去重实验结果（10 节点，1/3/6 步）

使用本项目 `fhf_dedup_gef2017/` 中的普通 FHF、HFF、HFFPS 已完成实验。
显示当前去重数据上的精度、训练记录、耗时和层级一致性。Naive 基线只保留一份。
全部统计从当前逐节点指标、预测 CSV 和训练残差文件重建，不启动模型训练。

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT = PROJECT_ROOT
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from PPHF_modules.results_check.current_results import (
    prepare_current_results, core_accuracy_table, plot_training, plot_timing,
)

In [ ]:
DATASET = 'gef2017'
SOURCE_DIR = ROOT / 'fhf_dedup_gef2017'
OUTPUT_DIR = ROOT / 'Results_check' / 'deduplication_results' / DATASET
RECOMPUTE = False  # 输入文件内容变化时会自动重建。True 强制重新计算。
print('Source:', SOURCE_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
tables = prepare_current_results(DATASET, root=ROOT, source_dir=SOURCE_DIR,
                                 output_dir=OUTPUT_DIR, force=RECOMPUTE)
display(tables['hierarchy_audit'])

## 精度比较

MAE、MASE、RMSSE 先逐节点计算，再在语义评估层级内等权平均。
去重后成为叶节点的序列归入底层，结构深度保留用于后续父子约束检查。
核心表包含 base、BU、MinT-shrinkage、WLSv 和共享 Naive；完整方法见 `accuracy_by_level.csv`。

In [ ]:
core = core_accuracy_table(tables)
display(core)
core.to_csv(OUTPUT_DIR / 'core_accuracy_by_level.csv')

## 全方法与 MCB 输入

下表保留全部协调方法。MCB 输入按预测步数和节点配对，两个 CSV 分别为
`mcb_mase.csv` 和 `mcb_rmsse.csv`。MCB 输入采用三个框架的 base、BU、MinT-shrinkage、WLSv
以及共享 Naive，共 13 种方法；完整精度表保留全部协调方法。

In [ ]:
display(tables['accuracy_by_level'])
print('MCB MASE shape:', tables['mcb_mase'].shape)
display(tables['mcb_mase'].head())

## 训练损失与耗时

图表使用源实验保存的日志和计时，不代表当前电脑重新运行的速度。
HFF/HFFPS 的轮数以 cloud round 为准。

In [ ]:
figure = plot_training(tables, OUTPUT_DIR)
plt.close(figure)
display(Image(filename=str(OUTPUT_DIR / 'training_loss.png')))
display(tables['timings'])
figure = plot_timing(tables, OUTPUT_DIR)
plt.close(figure)
display(Image(filename=str(OUTPUT_DIR / 'training_time_per_round.png')))

## 层级一致性

父子差异根据结构层级计算。MASI 用训练目标的一阶差分尺度归一化，父节点等权平均。
另列预测尺度和误差尺度归一化差异。数值一致性与预测精度分别报告。

In [ ]:
coherence = tables['coherence_by_level']
display(coherence.loc[coherence.method_internal.isin(['base','bu','mint_shrinkage','mint_var','naive']),
    ['horizon','source_system','method_internal','level','incoherence_total','MASI',
     'average_scale_free_incoherence_ratio_pct','average_error_scaled_incoherence_ratio_pct']])

In [ ]:
validation = json.loads((OUTPUT_DIR / 'validation.json').read_text())
display(pd.DataFrame([validation]))
print('Source fingerprints:', OUTPUT_DIR / 'source_manifest.json')
print('Generated tables:', sorted(p.name for p in OUTPUT_DIR.glob('*.csv')))